In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import bernoulli, binom

# ==============================================================================
# SUBPUNCTUL a) Simulare drum (traseu complet)
# ==============================================================================
def simuleaza_drum(nr_pasi, p):
    """
    Simulează o singură deplasare de 'nr_pasi'.
    p = probabilitatea de a merge la DREAPTA (+1).
    1-p = probabilitatea de a merge la STÂNGA (-1).
    """
    # 1. Generăm pașii (0 sau 1) folosind Bernoulli
    # bernoulli.rvs(p, size=N) returnează o listă de N valori de 0 și 1
    # 1 apare cu probabilitatea p, 0 cu probabilitatea 1-p
    pasi_raw = bernoulli.rvs(p, size=nr_pasi)

    # 2. Transformăm în deplasări (+1 sau -1)
    # Dacă pasul e 1 -> +1 (Dreapta)
    # Dacă pasul e 0 -> -1 (Stânga)
    deplasari = [1 if x == 1 else -1 for x in pasi_raw]

    # 3. Calculăm pozițiile cumulative (unde se află după fiecare pas)
    # Pornim de la 0. np.cumsum face suma parțială: [1, -1, 1] -> [1, 0, 1]
    pozitii = [0] + np.cumsum(deplasari).tolist()

    return pozitii

# Testăm funcția
print("a) Traseu generat:", simuleaza_drum(nr_pasi=10, p=0.5))


# ==============================================================================
# SUBPUNCTUL b) Histograma pozițiilor finale pe AXĂ
# ==============================================================================
def histograma_axa(N_sim, nr_pasi, p):
    """
    Simulăm de N_sim ori doar poziția finală și facem histograma.
    """
    # Metoda Rapidă (fără buclă for, vectorizat):
    # Generăm k (nr. succese/pași dreapta) pentru toate simulările deodată
    # k ~ Binomial(n, p)
    k_succese = binom.rvs(n=nr_pasi, p=p, size=N_sim)

    # Formula poziției finale:
    # Poz = (nr_dreapta * 1) + (nr_stanga * -1)
    # Poz = k - (n - k) = 2k - n
    pozitii_finale = 2 * k_succese - nr_pasi

    # Desenăm Histograma
    plt.figure(figsize=(8, 4))
    # 'bins' centrate pe întregi pentru claritate
    bins_range = range(min(pozitii_finale), max(pozitii_finale) + 2)

    plt.hist(pozitii_finale, bins=bins_range, align='left', rwidth=0.8,
             color='skyblue', edgecolor='black', alpha=0.7, label='Simulare')
    plt.title(f"Distribuția pozițiilor finale (Axă): n={nr_pasi}, p={p}")
    plt.xlabel("Poziția")
    plt.ylabel("Frecvența")
    plt.legend()
    plt.grid(axis='y', alpha=0.3)
    plt.show()

# Apelăm funcția
histograma_axa(N_sim=1000, nr_pasi=50, p=0.5)


# ==============================================================================
# SUBPUNCTUL c) Histograma pozițiilor finale pe CERC
# ==============================================================================
def histograma_cerc(N_sim, nr_pasi, p, n_noduri):
    """
    Aceeași idee, dar pozițiile sunt modulo n_noduri (0, 1, ..., n-1).
    """
    # Generăm k succese (pași dreapta)
    k_succese = binom.rvs(n=nr_pasi, p=p, size=N_sim)

    # Poziția pe axă (poate fi negativă sau > n_noduri)
    pozitii_axa = 2 * k_succese - nr_pasi

    # Transformăm pe cerc folosind operatorul modulo %
    # Ex: pe cerc cu 5 noduri, poziția 6 devine 1, poziția -1 devine 4.
    pozitii_cerc = pozitii_axa % n_noduri

    # Desenăm Histograma
    plt.figure(figsize=(8, 4))
    # Bins de la 0 la n_noduri
    plt.hist(pozitii_cerc, bins=range(n_noduri + 1), align='left', rwidth=0.8,
             color='lightgreen', edgecolor='black', alpha=0.7)
    plt.title(f"Distribuția pe Cerc ({n_noduri} noduri)")
    plt.xlabel("Nodul")
    # Forțăm axa X să arate toate nodurile
    plt.xticks(range(n_noduri))
    plt.grid(axis='y', alpha=0.3)
    plt.show()

# Apelăm funcția
histograma_cerc(N_sim=1000, nr_pasi=50, p=0.5, n_noduri=12)

In [ ]:
from scipy.stats import hypergeom, geom

# ==============================================================================
# Configurare Loto 6/49
# ==============================================================================
# M = Total bile în urnă (49)
# n = Bile "bune" (câștigătoare) într-o extragere (6)
# N = Bile extrase pe biletul meu (6)
M, n_bune, N_extrase = 49, 6, 6

# 1. Calculăm probabilitatea ca UN SINGUR bilet să fie câștigător
# Câștigător = are cel puțin 3 numere bune (3, 4, 5 sau 6)
# P(X >= 3) = 1 - P(X <= 2)
# Folosim hypergeom.cdf(k, M, n, N) care calculează P(X <= k)
prob_castig_un_bilet = 1 - hypergeom.cdf(2, M, n_bune, N_extrase)

print(f"Probabilitatea unui bilet câștigător (p): {prob_castig_un_bilet:.5f}")
# Ar trebui să fie aprox 0.0186 (1.8%)

# ==============================================================================
# SUBPUNCTUL i) Generare listă eșecuri
# ==============================================================================
# Vrem să simulăm de câte ori cumpărăm bilete NECÂȘTIGĂTOARE până dăm de unul bun.
# Aceasta este definiția distribuției Geometrice.
# geom.rvs(p) = numărul de încercări necesare pentru a obține primul succes (inclusiv succesul)
# Dacă vrem nr. de EȘECURI dinainte, scădem 1.

nr_simulari = 2000
# Generăm lista
incercari = geom.rvs(prob_castig_un_bilet, size=nr_simulari)
esecuri_pana_la_castig = incercari - 1

print(f"i) Media biletelor pierdute înainte de câștig: {np.mean(esecuri_pana_la_castig):.2f}")


# ==============================================================================
# SUBPUNCTUL ii) Estimare P(>= 10 eșecuri consecutive)
# ==============================================================================
# "Cel puțin 10 bilete succesive sunt necâștigătoare" înseamnă că primele 10 au fost eșecuri.
# Adică am avut nevoie de > 10 încercări pentru a câștiga.

# A. Estimare prin simulare (Monte Carlo)
# Numărăm de câte ori am avut >= 10 eșecuri în lista generată
cazuri_favorabile = sum(1 for x in esecuri_pana_la_castig if x >= 10)
prob_estimata = cazuri_favorabile / nr_simulari

# B. Valoare Teoretică
# Pentru Geometrică: P(Eșecuri >= k) = (1 - p)^k
# Adică probabilitatea să pierzi de k ori la rând.
prob_teoretica = (1 - prob_castig_un_bilet) ** 10

print(f"ii) Probabilitate (>=10 eșecuri):")
print(f"    Estimată: {prob_estimata:.4f}")
print(f"    Teoretică: {prob_teoretica:.4f}")